# Reflow-JEPA — Phase B: Synthetic-Data Training (Kaggle GPU)

Clones the project repo from GitHub, installs dependencies, and runs `train.py` on the procedural synthetic image/caption dataset at a realistic scale.

**Before running:** set `REPO_URL` below to your GitHub repo, and turn on GPU + Internet in the notebook's Settings panel (Internet is only required if you later set `USE_REAL_CHECKPOINTS = True`).

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/reflow-jepa.git"  # <-- set this
BRANCH = "main"
USE_REAL_CHECKPOINTS = False  # True needs internet access + real I-JEPA/T5 checkpoints reachable

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- turn on GPU in notebook Settings')

In [ ]:
!rm -rf /kaggle/working/repo
!git clone --branch {BRANCH} {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

## Sanity check: run the pre-implementation test suite first

43 tests, ~1-2 min on GPU-enabled Kaggle. If any of these fail, don't proceed to training -- something in the environment (transformers version, etc.) differs from what was validated.

In [ ]:
import os
os.environ['PYTHONPATH'] = '/kaggle/working/repo/src'
%cd /kaggle/working/repo/tests
!PYTHONPATH=/kaggle/working/repo/src python -m pytest -q

## Phase B training run

Full-scale config (6-layer predictor, deeper encoders than the local CPU smoke test). Adjust `--steps`/`--batch-size` to fit your GPU's memory and the time budget for this session.

In [ ]:
%cd /kaggle/working/repo/src

real_ckpt_flag = "--real-checkpoints" if USE_REAL_CHECKPOINTS else ""

!python train.py \
  --steps 3000 \
  --batch-size 32 \
  --lr 3e-4 \
  --sigma 0.3 \
  --vicreg-weight 5.0 \
  --recon-weight 1.0 \
  --predictor-depth 6 \
  --predictor-heads 8 \
  --visual-layers 4 \
  --text-layers 4 \
  --dataset-length 50000 \
  --eval-every 100 \
  --log-path /kaggle/working/training_log.json \
  --checkpoint-path /kaggle/working/reflow_jepa_ckpt.pt \
  {real_ckpt_flag} \
  --device cuda

## Diagnostics

Plots the quantities `DESIGN.md` §2.4 and the proof doc's evaluation protocol actually care about: the CFM loss (should decrease), the reconstruction loss (decoder learning to use the true embedding), the VICReg penalties (should trend toward 0, not stay pinned near `gamma_0^2`), and the manifold-adherence rate (Theorem 2's empirical counterpart -- the honest check of whether the trained flow is actually reaching the target manifold, not just assumed to per Assumption 2).

In [ ]:
import json
import matplotlib.pyplot as plt

with open('/kaggle/working/training_log.json') as f:
    log = json.load(f)

steps = [d['step'] for d in log]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(steps, [d['cfm_loss'] for d in log])
axes[0, 0].set_title('CFM loss'); axes[0, 0].set_xlabel('step')

axes[0, 1].plot(steps, [d['recon_loss'] for d in log])
axes[0, 1].axhline(y=10.4, color='gray', linestyle='--', label='near-random baseline (ln vocab)')
axes[0, 1].set_title('Reconstruction loss'); axes[0, 1].set_xlabel('step'); axes[0, 1].legend()

axes[1, 0].plot(steps, [d['vicreg_v'] for d in log], label='vicreg_v (visual)')
axes[1, 0].plot(steps, [d['vicreg_t'] for d in log], label='vicreg_t (text)')
axes[1, 0].axhline(y=1.0, color='gray', linestyle='--', label='gamma_0^2 (full collapse)')
axes[1, 0].set_title('VICReg penalties'); axes[1, 0].set_xlabel('step'); axes[1, 0].legend()

eval_steps = [d['step'] for d in log if 'eval_manifold_adherence_rate' in d]
eval_adherence = [d['eval_manifold_adherence_rate'] for d in log if 'eval_manifold_adherence_rate' in d]
axes[1, 1].plot(eval_steps, eval_adherence, marker='o')
axes[1, 1].set_title('Manifold adherence rate (eval)'); axes[1, 1].set_xlabel('step')
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('/kaggle/working/diagnostics.png', dpi=120)
plt.show()

## Qualitative check: decode a few generated captions

With the mock tokenizer (`--real-checkpoints` off), decoded token ids won't map back to readable English -- only useful with real checkpoints. Included for when you flip `USE_REAL_CHECKPOINTS = True`.

In [ ]:
import torch
from reflow_jepa import ReflowJEPA
from synthetic_data import SyntheticCaptioningDataset, collate_images_captions
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ReflowJEPA(
    predictor_depth=6, predictor_heads=8, visual_layers=4, text_layers=4,
    real_checkpoints=USE_REAL_CHECKPOINTS,
).to(device)
model.load_state_dict(torch.load('/kaggle/working/reflow_jepa_ckpt.pt', map_location=device))
model.eval()

ds = SyntheticCaptioningDataset(length=8, seed=1234)
dl = DataLoader(ds, batch_size=8, collate_fn=collate_images_captions)
images, true_captions = next(iter(dl))
images = images.to(device)

generated_ids = model.generate_captions(images, max_new_tokens=16, n_steps=50)

for i in range(len(true_captions)):
    print(f'true: {true_captions[i]}')
    if USE_REAL_CHECKPOINTS:
        print(f'gen : {model.tokenizer.decode(generated_ids[i], skip_special_tokens=True)}')
    else:
        print(f'gen (token ids, mock tokenizer -- not meaningful English): {generated_ids[i].tolist()}')
    print()